# Faruq-v3 — paired AF2 and IGEM1 confirmation

Validation-only seed 42/123/2026. Seed 42 dan D0FT tiga-seed digunakan kembali; notebook hanya melatih AF2 dan IGEM1 seed 123/2026. Output persisten berada pada satu folder Drive, dapat di-resume lintas akun, dan test tidak diekstrak.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/af2-igem-paired-confirmation'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO, check=True)
else:
    if REPO.exists():
        shutil.rmtree(REPO)
    clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
    for attempt in range(1, 4):
        result = subprocess.run(clone)
        if result.returncode == 0:
            break
        if REPO.exists():
            shutil.rmtree(REPO)
        if attempt == 3:
            raise RuntimeError('Git clone gagal tiga kali.')
        time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for key in list(sys.modules):
    if key == 'coffee_detector' or key.startswith('coffee_detector.'):
        sys.modules.pop(key, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REQUIRED = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/val_reports/lfdet_afab_seed42_screening.json',
    'experiments/faruq-v3-breadth-screening-batch-v1/candidates/IGEM/val_reports/igem_seed42_screening.json',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/val_reports/acmc1_paired_optimization_confirmation.json',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed2026/weights/best.pt',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
AF2_SEED42 = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
IGEM_SEED42 = require_project_artifact(PROJECT_ROOT, REQUIRED[2])
D0FT_CONFIRMATION = require_project_artifact(PROJECT_ROOT, REQUIRED[3])
D0_123 = require_project_artifact(PROJECT_ROOT, REQUIRED[4])
D0_2026 = require_project_artifact(PROJECT_ROOT, REQUIRED[5])
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-af2-igem-paired-confirmation-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU    :', torch.cuda.get_device_name(0))
print('PROJECT:', PROJECT_ROOT)
print('OUTPUT :', OUTPUT_ROOT)
for seed in (123, 2026):
    for arm in ('AF2', 'IGEM1'):
        run_dir = OUTPUT_ROOT / arm / f'{arm}_seed{seed}'
        csv_path = run_dir / 'results.csv'
        epochs = 0
        if csv_path.is_file():
            import pandas as pd
            epochs = len(pd.read_csv(csv_path))
        complete = (run_dir / 'weights/best.pt').is_file() and epochs >= 50
        print(f'{arm} seed {seed}: ' + ('COMPLETE' if complete else f'{epochs}/50'))

## Jalankan konfirmasi

Empat run berjalan berurutan dengan log penuh dialihkan dari browser. Jika runtime terputus, jalankan kembali notebook dari awal; run lengkap dilewati dan run parsial dilanjutkan dari `last.pt`. Jangan membuka runtime kedua pada output yang sama secara bersamaan.

In [ ]:
import csv

command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_af2_igem_paired_confirmation',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--af2-seed42-result', str(AF2_SEED42),
    '--igem-seed42-result', str(IGEM_SEED42),
    '--d0ft-confirmation', str(D0FT_CONFIRMATION),
    '--d0-checkpoints', str(D0_123), str(D0_2026),
    '--output-root', str(OUTPUT_ROOT),
    '--seeds', '123', '2026',
    '--models', 'AF2', 'IGEM1',
    '--device', '0', '--authorize-training',
]
print('MENJALANKAN AF2/IGEM1 paired confirmation tanpa progress bar browser.', flush=True)
RUN_LOG = Path('/content/af2_igem_paired_confirmation.log')
with RUN_LOG.open('a', encoding='utf-8', buffering=1) as log_stream:
    process = subprocess.Popen(
        command, cwd=REPO, text=True, stdout=log_stream, stderr=subprocess.STDOUT
    )
    while process.poll() is None:
        statuses = []
        for seed in (123, 2026):
            for arm in ('AF2', 'IGEM1'):
                csv_path = OUTPUT_ROOT / arm / f'{arm}_seed{seed}/results.csv'
                epochs = 0
                if csv_path.is_file():
                    try:
                        with csv_path.open(newline='', encoding='utf-8') as stream:
                            epochs = sum(1 for _ in csv.DictReader(stream))
                    except Exception:
                        epochs = 0
                statuses.append(f'{arm}-{seed}={epochs}/50')
        print('[STATUS]', ', '.join(statuses), flush=True)
        time.sleep(60)
    return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(RUN_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-120:])
    print(tail)
    raise RuntimeError(f'Konfirmasi AF2/IGEM1 gagal, return code={return_code}; log={RUN_LOG}')
print('TRAINING DAN EVALUASI SELESAI. Log lengkap:', RUN_LOG)

In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/af2_igem_paired_confirmation.json'
assert SUMMARY.is_file(), f'Konfirmasi belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
rows = []
for model, metrics in result['aggregate'].items():
    for metric, values in metrics.items():
        rows.append({'model': model, 'metric': metric, **values})
percentage = ('d0ft_mean', 'd0ft_std', 'candidate_mean', 'candidate_std', 'head_delta_mean', 'head_delta_std', 'head_delta_min')
display(pd.DataFrame(rows).style.format({key: '{:.2%}' for key in percentage}))
print('PER-SEED DELTAS:')
for model, metrics in result['aggregate'].items():
    for metric, values in metrics.items():
        print(model, metric, values['deltas'])
print('DECISIONS:', result['decisions'])
print('STATUS   :', result['status'])
print('NEXT     :', result['next_action'])
print('SUMMARY  :', SUMMARY)
print('Kirim tabel dan keputusan. Test tidak dibuka.')